<a href="https://colab.research.google.com/github/isaac-sun/fedfree/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# FedFree — Federated Learning Free-Rider Defense

**DistilBERT + PEFT LoRA + Per-Class Shapley Detection on Yahoo Answers**

Two-phase defense: Positive-Sum Filter → Variance Fingerprinting

## 1. Setup

Clone the repository.

In [ ]:
import os, sys

REPO_NAME = "fedfree"
GITHUB_USER = "isaac-sun"

if os.path.basename(os.getcwd()) == REPO_NAME:
    print("Already in fedfree/ — pulling latest...")
    !git pull origin main
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}
    !git pull origin main
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_NAME}

print(f"Working directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
# Install dependencies (no datasets needed — we use huggingface_hub directly)
!pip install 'huggingface-hub>=0.20' 'torchao>=0.16.0' torch transformers peft scikit-learn numpy pandas pyarrow matplotlib seaborn tqdm
print("Dependencies installed.")

## 3. Verify GPU

In [ ]:
import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM:     {vram:.1f} GB")
else:
    print("Running on CPU — may be slow.")

## 4. Verify Imports

In [ ]:
# Quick sanity check — if this fails, check the install cell above
import torch, pandas, numpy
from huggingface_hub import hf_hub_download, list_repo_files
from peft import LoraConfig
import torchao; print(f"  torchao {torchao.__version__} ✓")
print("All critical imports OK ✓")

## 5. Run Experiments

Four experiments: **baseline**, **DFR**, **SDFR**, **AFR** (same as 20NEWS-FL).

Expected runtime:
- **L4/T4 GPU**: ~2–3 hours
- **A100 GPU**: ~1–1.5 hours
- **CPU**: ~8–10 hours

In [ ]:
import subprocess, os, sys
# Clean stale bytecode
subprocess.run(["find", ".", "-type", "d", "-name", "__pycache__", "-exec", "rm", "-rf", "{}", "+"], capture_output=True)
print("__pycache__ cleaned")
# Purge ALL fedfree modules from cache so %run picks up fresh code
_fedfree_mods = [m for m in sorted(sys.modules) if any(
    m == p or m.startswith(p + ".") for p in
    ("config", "attacks", "models", "fl", "defense", "data", "utils", "main")
)]
for _m in _fedfree_mods:
    del sys.modules[_m]
print(f"Purged {len(_fedfree_mods)} cached modules: {_fedfree_mods}")
%run main.py

## 5. Convergence Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

csv_path = "results/f1_curves.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, index_col=0)
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = {"baseline_no_attack": "#4D4D4D", "attack_dfr": "#B2182B",
              "attack_sdfr": "#2166AC", "attack_afr": "#1B7837"}
    for col in df.columns:
        c = colors.get(col, None)
        ax.plot(df.index, df[col], label=col, color=c, linewidth=1.5)
    ax.set_xlabel("Round")
    ax.set_ylabel("Macro-F1")
    ax.set_title("FedFree — Attack Comparison (Yahoo Answers, DistilBERT+LoRA)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    os.makedirs("results/plots", exist_ok=True)
    fig.savefig("results/plots/convergence.pdf", dpi=300, bbox_inches="tight")
    plt.show()
    print("Convergence plot saved to results/plots/convergence.pdf")
else:
    print("f1_curves.csv not found — run main.py first")

## 6. View Results

In [ ]:
import pandas as pd
import os

csv_path = "results/f1_curves.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, index_col=0)
    print(f"Experiments: {list(df.columns)}")
    print(f"Rounds: {len(df)}")
    print()
    baseline = df["baseline_no_attack"].iloc[-1] if "baseline_no_attack" in df.columns else None
    print(f"{'Experiment':<25s} {'Final F1':>10s} {'Δ Baseline':>12s}")
    print("-" * 50)
    for col in df.columns:
        final = df[col].iloc[-1]
        delta = final - baseline if baseline else float("nan")
        d_str = f"{delta:+.4f}" if delta == delta else "—"
        print(f"  {col:<23s} {final:>10.4f} {d_str:>12s}")
else:
    print("f1_curves.csv not found — run main.py first")

# Show convergence plot
import matplotlib.pyplot as plt
plot_path = "results/plots/convergence.pdf"
if os.path.exists(plot_path):
    from IPython.display import Image as IPImage
    # Convert first page of PDF to show
    print("\n### Convergence Curves")
    print(f"See: {plot_path}")

## 7. Download Results

In [ ]:
!zip -r fedfree_results.zip results/ 2>/dev/null
try:
    from google.colab import files
    files.download("fedfree_results.zip")
except ImportError:
    print("fedfree_results.zip ready for manual download.")